In [1]:
# ============================================================
#    CHESS  
# ============================================================


import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ============================================================
# STEP 1 - DEFINE THE PIECES
# ============================================================
# Each piece is a letter:
#   Uppercase = WHITE pieces (K Q R B N P)
#   Lowercase = BLACK pieces (k q r b n p)
# K = King, Q = Queen, R = Rook, B = Bishop, N = Knight, P = Pawn

# These are the chess symbols (emojis) for each piece
SYMBOLS = {
    'K': '♔', 'Q': '♕', 'R': '♖', 'B': '♗', 'N': '♘', 'P': '♙',
    'k': '♚', 'q': '♛', 'r': '♜', 'b': '♝', 'n': '♞', 'p': '♟',
    ' ': '  '   # empty square
}

# ============================================================
# STEP 2 - CREATE THE STARTING BOARD
# ============================================================
# The board is a list of 8 rows, each row has 8 squares.
# Row 0 = top (Black's back row), Row 7 = bottom (White's back row)

def create_board():
    """Returns the starting chess board as a list of lists."""
    return [
        ['r','n','b','q','k','b','n','r'],   # row 0: black pieces
        ['p','p','p','p','p','p','p','p'],   # row 1: black pawns
        [' ',' ',' ',' ',' ',' ',' ',' '], # row 2: empty
        [' ',' ',' ',' ',' ',' ',' ',' '], # row 3: empty
        [' ',' ',' ',' ',' ',' ',' ',' '], # row 4: empty
        [' ',' ',' ',' ',' ',' ',' ',' '], # row 5: empty
        ['P','P','P','P','P','P','P','P'],   # row 6: white pawns
        ['R','N','B','Q','K','B','N','R'],   # row 7: white pieces
    ]

# ============================================================
# STEP 3 - HELPER FUNCTIONS
# ============================================================

def is_white(piece):
    """Returns True if the piece is White (uppercase letter)."""
    return piece != ' ' and piece.isupper()

def is_black(piece):
    """Returns True if the piece is Black (lowercase letter)."""
    return piece != ' ' and piece.islower()

def is_empty(piece):
    """Returns True if the square is empty."""
    return piece == ' '

def is_on_board(row, col):
    """Returns True if (row, col) is inside the 8x8 board."""
    return 0 <= row <= 7 and 0 <= col <= 7

def is_enemy(piece, white_turn):
    """Returns True if piece belongs to the opponent."""
    if white_turn:
        return is_black(piece)
    else:
        return is_white(piece)

def is_friend(piece, white_turn):
    """Returns True if piece belongs to the current player."""
    if white_turn:
        return is_white(piece)
    else:
        return is_black(piece)

# ============================================================
# STEP 4 - FIND VALID MOVES FOR EACH PIECE
# ============================================================

def get_moves(board, row, col, check_safety=True):
    """
    Returns a list of (row, col) squares the piece at (row,col) can move to.
    check_safety=True means we filter out moves that leave our king in check.
    """
    piece = board[row][col]
    if is_empty(piece):
        return []

    white = is_white(piece)
    kind  = piece.lower()   # get type ignoring colour
    moves = []

    # --- PAWN ---
    if kind == 'p':
        direction = -1 if white else 1          # white moves up (row decreases)
        start_row = 6    if white else 1         # pawns start here

        # Move forward 1 square (only if empty)
        r1 = row + direction
        if is_on_board(r1, col) and is_empty(board[r1][col]):
            moves.append((r1, col))
            # Move forward 2 squares from starting position
            r2 = row + 2 * direction
            if row == start_row and is_empty(board[r2][col]):
                moves.append((r2, col))

        # Capture diagonally (only if enemy piece is there)
        for dc in [-1, 1]:
            rc, cc = row + direction, col + dc
            if is_on_board(rc, cc) and is_enemy(board[rc][cc], white):
                moves.append((rc, cc))

    # --- KNIGHT (moves in L-shape, can jump over pieces) ---
    elif kind == 'n':
        jumps = [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]
        for dr, dc in jumps:
            r, c = row + dr, col + dc
            if is_on_board(r, c) and not is_friend(board[r][c], white):
                moves.append((r, c))

    # --- ROOK (slides in straight lines) ---
    elif kind == 'r':
        for dr, dc in [(0,1),(0,-1),(1,0),(-1,0)]:
            r, c = row + dr, col + dc
            while is_on_board(r, c):
                if is_friend(board[r][c], white):
                    break               # blocked by own piece
                moves.append((r, c))
                if is_enemy(board[r][c], white):
                    break               # can capture, but can't go further
                r += dr; c += dc

    # --- BISHOP (slides diagonally) ---
    elif kind == 'b':
        for dr, dc in [(1,1),(1,-1),(-1,1),(-1,-1)]:
            r, c = row + dr, col + dc
            while is_on_board(r, c):
                if is_friend(board[r][c], white):
                    break
                moves.append((r, c))
                if is_enemy(board[r][c], white):
                    break
                r += dr; c += dc

    # --- QUEEN (rook + bishop combined) ---
    elif kind == 'q':
        for dr, dc in [(0,1),(0,-1),(1,0),(-1,0),(1,1),(1,-1),(-1,1),(-1,-1)]:
            r, c = row + dr, col + dc
            while is_on_board(r, c):
                if is_friend(board[r][c], white):
                    break
                moves.append((r, c))
                if is_enemy(board[r][c], white):
                    break
                r += dr; c += dc

    # --- KING (one step in any direction) ---
    elif kind == 'k':
        for dr, dc in [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]:
            r, c = row + dr, col + dc
            if is_on_board(r, c) and not is_friend(board[r][c], white):
                moves.append((r, c))

    # Filter: remove moves that leave OUR king in check
    if check_safety:
        safe_moves = []
        for (tr, tc) in moves:
            # Simulate the move on a copy of the board
            test = [row_[:] for row_ in board]
            test[tr][tc] = piece
            test[row][col] = ' '
            if not is_in_check(test, white):
                safe_moves.append((tr, tc))
        return safe_moves

    return moves

# ============================================================
# STEP 5 - CHECK DETECTION
# ============================================================

def is_in_check(board, white):
    """
    Returns True if the king (of the given colour) is in check.
    Checks whether any enemy piece can attack the king.
    """
    # Find the king's position
    king = 'K' if white else 'k'
    king_pos = None
    for r in range(8):
        for c in range(8):
            if board[r][c] == king:
                king_pos = (r, c)

    if king_pos is None:
        return True  # king is gone = definitely in check

    kr, kc = king_pos

    # Check if any enemy piece can reach the king
    for r in range(8):
        for c in range(8):
            piece = board[r][c]
            if is_enemy(piece, white):
                # Get moves WITHOUT safety check (avoid infinite loop)
                enemy_moves = get_moves(board, r, c, check_safety=False)
                if (kr, kc) in enemy_moves:
                    return True   # king is attacked!
    return False

def has_any_moves(board, white):
    """Returns True if the player (white or black) has at least one legal move."""
    for r in range(8):
        for c in range(8):
            piece = board[r][c]
            if (white and is_white(piece)) or (not white and is_black(piece)):
                if len(get_moves(board, r, c)) > 0:
                    return True
    return False

# ============================================================
# STEP 6 - DISPLAY THE BOARD (with colours using HTML)
# ============================================================

def render_board(board, selected=None, valid_moves=None, last_from=None, last_to=None, in_check_king=None):
    """
    Draws the chess board as a coloured HTML table inside Jupyter.
    - selected      : (row, col) of the selected piece
    - valid_moves   : list of (row, col) squares the piece can go
    - last_from/to  : squares from the last move (shown in yellow)
    - in_check_king : (row, col) of the king in check (shown in red)
    """
    if valid_moves is None:
        valid_moves = []

    # Column labels (a–h)
    files = ['a','b','c','d','e','f','g','h']

    html = """
    <style>
      .chess-table { border-collapse: collapse; font-size: 2rem; }
      .chess-table td { width: 60px; height: 60px; text-align: center;
                        vertical-align: middle; cursor: pointer; border: 1px solid #888; }
      .label-cell  { font-size: 0.85rem; color: #555; font-weight: bold;
                     width: 30px; height: 30px; text-align: center; }
    </style>
    <table class="chess-table">
    """

    # Top file labels (a-h)
    html += '<tr><td class="label-cell"></td>'
    for f in files:
        html += f'<td class="label-cell">{f}</td>'
    html += '</tr>'

    for r in range(8):
        html += f'<tr><td class="label-cell">{8 - r}</td>'  # rank label on left
        for c in range(8):
            piece = board[r][c]
            symbol = SYMBOLS.get(piece, '?')

            # Decide background colour
            is_light = (r + c) % 2 == 0
            bg = '#F0D9B5' if is_light else '#B58863'    # classic chess colours

            if selected and (r, c) == selected:
                bg = '#F6F669'                            # bright yellow = selected
            elif last_from and (r, c) == last_from:
                bg = '#CDD26A'                            # pale yellow = last move from
            elif last_to and (r, c) == last_to:
                bg = '#AAA23A'                            # darker yellow = last move to
            elif in_check_king and (r, c) == in_check_king:
                bg = '#FF6B6B'                            # red = king in check!
            elif (r, c) in valid_moves:
                if not is_empty(piece):
                    bg = '#E06060'                        # reddish = capture available
                else:
                    bg = '#7FC97F' if is_light else '#5AA55A'  # green = can move here

            html += f'<td style="background:{bg};">{symbol}</td>'
        html += f'<td class="label-cell">{8 - r}</td></tr>'  # rank label on right

    # Bottom file labels
    html += '<tr><td class="label-cell"></td>'
    for f in files:
        html += f'<td class="label-cell">{f}</td>'
    html += '</tr></table>'

    display(HTML(html))

# ============================================================
# STEP 7 - THE MAIN GAME (interactive with dropdowns)
# ============================================================

class ChessGame:
    """
    The main chess game class.
    Runs an interactive game inside Jupyter using dropdown menus.
    """

    def __init__(self):
        self.board       = create_board()   # start fresh board
        self.white_turn  = True             # white always goes first
        self.selected    = None             # which square is selected
        self.valid_moves = []               # valid squares for selected piece
        self.last_from   = None             # last move: from square
        self.last_to     = None             # last move: to square
        self.captured_w  = []              # pieces captured BY white
        self.captured_b  = []              # pieces captured BY black
        self.move_count  = 0               # total moves made
        self.game_over   = False
        self.setup_ui()

    # ---- BUILD THE WIDGETS ----
    def setup_ui(self):
        """Creates all the buttons and dropdowns."""

        files = ['a','b','c','d','e','f','g','h']
        ranks = ['8','7','6','5','4','3','2','1']
        squares = [f + r for r in ranks for f in files]   # a8, b8, ... h1

        style = {'description_width': '80px'}
        layout = widgets.Layout(width='160px')

        # Dropdown: choose FROM square
        self.from_dd = widgets.Dropdown(
            options=['--'] + squares,
            description='From:',
            style=style, layout=layout
        )

        # Dropdown: choose TO square
        self.to_dd = widgets.Dropdown(
            options=['--'] + squares,
            description='To:',
            style=style, layout=layout
        )

        # Buttons
        self.preview_btn = widgets.Button(description='👁 Preview Moves',
                                          button_style='info',
                                          layout=widgets.Layout(width='140px'))
        self.move_btn    = widgets.Button(description='✅ Make Move',
                                          button_style='success',
                                          layout=widgets.Layout(width='130px'))
        self.reset_btn   = widgets.Button(description='🔄 New Game',
                                          button_style='warning',
                                          layout=widgets.Layout(width='120px'))

        # Output areas
        self.board_out  = widgets.Output()   # board goes here
        self.status_out = widgets.Output()   # status message goes here

        # Connect buttons to functions
        self.preview_btn.on_click(self.on_preview)
        self.move_btn.on_click(self.on_move)
        self.reset_btn.on_click(self.on_reset)

        # Layout
        controls = widgets.HBox([self.from_dd, self.to_dd,
                                  self.preview_btn, self.move_btn, self.reset_btn])

        # Show everything
        display(widgets.VBox([controls, self.status_out, self.board_out]))
        self.draw()

    # ---- HELPER: convert "e2" → (row, col) ----
    def sq_to_rc(self, sq):
        """Converts a square name like 'e2' to (row, col) numbers."""
        files = 'abcdefgh'
        col = files.index(sq[0])       # 'a'=0, 'b'=1, ... 'h'=7
        row = 8 - int(sq[1])           # '8'=0, '7'=1, ... '1'=7
        return (row, col)

    def rc_to_sq(self, row, col):
        """Converts (row, col) numbers back to square name like 'e2'."""
        files = 'abcdefgh'
        return files[col] + str(8 - row)

    # ---- DRAW BOARD + STATUS ----
    def draw(self):
        """Redraws the board and status message."""
        # Find king position if in check
        in_check_king = None
        if not self.game_over:
            if is_in_check(self.board, self.white_turn):
                king = 'K' if self.white_turn else 'k'
                for r in range(8):
                    for c in range(8):
                        if self.board[r][c] == king:
                            in_check_king = (r, c)

        with self.board_out:
            clear_output(wait=True)
            render_board(self.board,
                         selected=self.selected,
                         valid_moves=self.valid_moves,
                         last_from=self.last_from,
                         last_to=self.last_to,
                         in_check_king=in_check_king)

    def show_status(self, msg, color='black'):
        """Shows a status message below the controls."""
        with self.status_out:
            clear_output(wait=True)
            display(HTML(f'<p style="font-size:1rem; color:{color}; margin:6px 0;">{msg}</p>'))

    # ---- BUTTON: PREVIEW MOVES ----
    def on_preview(self, _):
        """When the user clicks 'Preview Moves', highlight valid moves."""
        if self.game_over:
            self.show_status('Game is over! Click New Game.', 'red')
            return

        sq = self.from_dd.value
        if sq == '--':
            self.show_status(' Please choose a FROM square first.', 'orange')
            return

        row, col = self.sq_to_rc(sq)
        piece = self.board[row][col]

        # Check it's the right player's piece
        if is_empty(piece):
            self.show_status(f' Square {sq} is empty! Pick a square with your piece.', 'orange')
            return
        if self.white_turn and not is_white(piece):
            self.show_status(" That's Black's piece! It's White's turn.", 'orange')
            return
        if not self.white_turn and not is_black(piece):
            self.show_status(" That's White's piece! It's Black's turn.", 'orange')
            return

        # Get valid moves
        self.selected    = (row, col)
        self.valid_moves = get_moves(self.board, row, col)
        self.draw()

        symbol = SYMBOLS[piece]
        count  = len(self.valid_moves)
        move_names = [self.rc_to_sq(r, c) for r, c in self.valid_moves]
        if count == 0:
            self.show_status(f'{symbol} at {sq} has NO legal moves!', 'red')
        else:
            self.show_status(
                f'{symbol} at {sq} can move to: {", ".join(move_names)} ({count} options) — '
                'Green = move, Red = capture', 'green')

    # ---- BUTTON: MAKE MOVE ----
    def on_move(self, _):
        """When the user clicks 'Make Move', execute the move."""
        if self.game_over:
            self.show_status('Game is over! Click New Game.', 'red')
            return

        from_sq = self.from_dd.value
        to_sq   = self.to_dd.value

        if from_sq == '--' or to_sq == '--':
            self.show_status(' Choose both FROM and TO squares!', 'orange')
            return
        if from_sq == to_sq:
            self.show_status(' FROM and TO squares cannot be the same!', 'orange')
            return

        fr, fc = self.sq_to_rc(from_sq)
        tr, tc = self.sq_to_rc(to_sq)
        piece  = self.board[fr][fc]

        # Validate piece ownership
        if is_empty(piece):
            self.show_status(f' No piece on {from_sq}!', 'orange')
            return
        if self.white_turn and not is_white(piece):
            self.show_status(" It's White's turn!", 'orange')
            return
        if not self.white_turn and not is_black(piece):
            self.show_status(" It's Black's turn!", 'orange')
            return

        # Check move is legal
        legal = get_moves(self.board, fr, fc)
        if (tr, tc) not in legal:
            self.show_status(f' Illegal move! {SYMBOLS[piece]} cannot go from {from_sq} to {to_sq}.', 'red')
            return

        # ---- EXECUTE THE MOVE ----
        captured = self.board[tr][tc]

        self.board[tr][tc] = piece
        self.board[fr][fc] = ' '

        # Pawn promotion: pawn reaches the other end → becomes Queen!
        if piece == 'P' and tr == 0:
            self.board[tr][tc] = 'Q'
            self.show_status(' Pawn promoted to Queen!', 'purple')
        if piece == 'p' and tr == 7:
            self.board[tr][tc] = 'q'
            self.show_status(' Pawn promoted to Queen!', 'purple')

        # Record captures
        if not is_empty(captured):
            if self.white_turn:
                self.captured_b.append(captured)
            else:
                self.captured_w.append(captured)

        # Update last move squares
        self.last_from = (fr, fc)
        self.last_to   = (tr, tc)
        self.move_count += 1

        # Clear selection
        self.selected    = None
        self.valid_moves = []

        # Switch turns
        self.white_turn = not self.white_turn
        next_player = 'White' if self.white_turn else 'Black'

        # Check for check / checkmate / stalemate
        in_check = is_in_check(self.board, self.white_turn)
        any_moves = has_any_moves(self.board, self.white_turn)

        self.draw()

        if not any_moves:
            if in_check:
                winner = 'Black' if self.white_turn else 'White'
                self.show_status(f' CHECKMATE! {winner} wins! Congratulations! 🎉', 'darkgreen')
            else:
                self.show_status(' STALEMATE! It\'s a draw — no legal moves but king is safe.', 'blue')
            self.game_over = True
        elif in_check:
            self.show_status(f' CHECK! {next_player}\'s King is in danger! Must escape!', 'red')
        else:
            cap_info = ''
            if not is_empty(captured):
                cap_info = f' Captured {SYMBOLS[captured]}!'
            self.show_status(f'✅ Moved {SYMBOLS[piece]} {from_sq}→{to_sq}.{cap_info} Now {next_player}\'s turn.', 'black')

    # ---- BUTTON: RESET ----
    def on_reset(self, _):
        """Starts a completely new game."""
        self.board       = create_board()
        self.white_turn  = True
        self.selected    = None
        self.valid_moves = []
        self.last_from   = None
        self.last_to     = None
        self.captured_w  = []
        self.captured_b  = []
        self.move_count  = 0
        self.game_over   = False
        self.from_dd.value = '--'
        self.to_dd.value   = '--'
        self.draw()
        self.show_status('New game started! White goes first. Choose a FROM square and click Preview!', 'navy')


# ============================================================
# STEP 8 - START THE GAME!
# ============================================================
print(" Welcome to Chess Adventure!")
print("=" * 40)
print("HOW TO PLAY:")
print("  1. Choose a FROM square (e.g. 'e2') from the dropdown")
print("  2. Click '👁 Preview Moves' to see where your piece can go")
print("  3. Choose a TO square from the second dropdown")
print("  4. Click '✅ Make Move' to move your piece")
print("  5. Click '🔄 New Game' to start over anytime")
print()
print("  Green squares = you can move there")
print("  Red squares   = you can capture there")
print("  Yellow square = your selected piece")
print("=" * 40)
print()

game = ChessGame()

 Welcome to Chess Adventure!
HOW TO PLAY:
  1. Choose a FROM square (e.g. 'e2') from the dropdown
  2. Click '👁 Preview Moves' to see where your piece can go
  3. Choose a TO square from the second dropdown
  4. Click '✅ Make Move' to move your piece
  5. Click '🔄 New Game' to start over anytime

  Green squares = you can move there
  Red squares   = you can capture there
  Yellow square = your selected piece

